# Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# Dataset Load

In [ ]:
# Load shift datasets
shift_train_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_train_seen_attacks.csv")
shift_val_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_val_seen_attacks.csv")
shift_test_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_test_unseen_attacks.csv")

print("Shift train:", shift_train_df.shape)
print("Shift val:", shift_val_df.shape)
print("Shift test:", shift_test_df.shape)

print("\nShift test label distribution:")
print(shift_test_df["binary_label"].value_counts())

print("\nShift test attack family distribution:")
print(shift_test_df["attack_family"].value_counts())

# Feature Selection and Training

In [ ]:
non_feature_cols = [
    "source_file",
    "attack_type",
    "attack_family",
    "binary_label"
]

feature_cols_shift = [
    col for col in shift_train_df.columns
    if col not in non_feature_cols
]

X_shift_train = shift_train_df[feature_cols_shift]
y_shift_train = shift_train_df["binary_label"]

X_shift_val = shift_val_df[feature_cols_shift]
y_shift_val = shift_val_df["binary_label"]

X_shift_test = shift_test_df[feature_cols_shift]
y_shift_test = shift_test_df["binary_label"]

print("Number of features:", len(feature_cols_shift))
print(X_shift_train.shape)
print(X_shift_val.shape)
print(X_shift_test.shape)

# XGBoost Training

In [ ]:
xgb_shift = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_shift.fit(X_shift_train, y_shift_train)

y_val_pred = xgb_shift.predict(X_shift_val)
y_val_prob = xgb_shift.predict_proba(X_shift_val)[:, 1]
print("Shift Validation Metrics:")
print("Accuracy:", accuracy_score(y_shift_val, y_val_pred))
print("Precision:", precision_score(y_shift_val, y_val_pred))
print("Recall:", recall_score(y_shift_val, y_val_pred))
print("F1:", f1_score(y_shift_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_shift_val, y_val_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_shift_val, y_val_pred))

y_test_pred = xgb_shift.predict(X_shift_test)
y_test_prob = xgb_shift.predict_proba(X_shift_test)[:, 1]

print("\n\nUnseen Attack-Family Shift Test Metrics:")
print("Accuracy:", accuracy_score(y_shift_test, y_test_pred))
print("Precision:", precision_score(y_shift_test, y_test_pred))
print("Recall:", recall_score(y_shift_test, y_test_pred))
print("F1:", f1_score(y_shift_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_shift_test, y_test_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_shift_test, y_test_pred))

# Recall By Unseen Attack Family

In [ ]:
shift_test_df["predicted_label"] = y_test_pred

print("Recall by unseen attack family:")

for family in shift_test_df["attack_family"].unique():
    family_df = shift_test_df[shift_test_df["attack_family"] == family]

    if family == "Benign":
        continue

    correct_attacks = ((family_df["binary_label"] == 1) & (family_df["predicted_label"] == 1)).sum()
    total_attacks = (family_df["binary_label"] == 1).sum()

    recall = correct_attacks / total_attacks

    print(family, ":", recall, "(", correct_attacks, "/", total_attacks, ")")

# SHAP for Attack-Family Shift Test

In [ ]:
# Use 5000 samples for SHAP
X_shift_shap = X_shift_test.sample(n=5000, random_state=42)

print("SHAP sample shape:", X_shift_shap.shape)

explainer_shift = shap.TreeExplainer(xgb_shift)
shap_values_shift = explainer_shift.shap_values(X_shift_shap)


shap_importance_shift = pd.DataFrame({
    "feature": X_shift_shap.columns,
    "mean_abs_shap": np.abs(shap_values_shift).mean(axis=0)
})

shap_importance_shift = shap_importance_shift.sort_values(
    by="mean_abs_shap",
    ascending=False
)

print("Top 20 SHAP features under attack-family shift:")
print(shap_importance_shift.head(20))


plt.figure()
shap.summary_plot(
    shap_values_shift,
    X_shift_shap,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(
    shap_values_shift,
    X_shift_shap,
    show=False
)
plt.tight_layout()
plt.show()

# Behavior-group SHAP contribution under shift

In [ ]:
feature_groups = {
    "Header_Length": "Packet-size behavior",
    "Protocol Type": "Protocol behavior",
    "Duration": "Timing/rate behavior",
    "Rate": "Timing/rate behavior",
    "Srate": "Timing/rate behavior",
    "Drate": "Timing/rate behavior",

    "fin_flag_number": "TCP flag/count behavior",
    "syn_flag_number": "TCP flag/count behavior",
    "rst_flag_number": "TCP flag/count behavior",
    "psh_flag_number": "TCP flag/count behavior",
    "ack_flag_number": "TCP flag/count behavior",
    "ece_flag_number": "TCP flag/count behavior",
    "cwr_flag_number": "TCP flag/count behavior",
    "ack_count": "TCP flag/count behavior",
    "syn_count": "TCP flag/count behavior",
    "fin_count": "TCP flag/count behavior",
    "rst_count": "TCP flag/count behavior",

    "HTTP": "Protocol behavior",
    "HTTPS": "Protocol behavior",
    "DNS": "Protocol behavior",
    "Telnet": "Protocol behavior",
    "SMTP": "Protocol behavior",
    "SSH": "Protocol behavior",
    "IRC": "Protocol behavior",
    "TCP": "Protocol behavior",
    "UDP": "Protocol behavior",
    "DHCP": "Protocol behavior",
    "ARP": "Protocol behavior",
    "ICMP": "Protocol behavior",
    "IGMP": "Protocol behavior",
    "IPv": "Protocol behavior",
    "LLC": "Protocol behavior",

    "Tot sum": "Packet-size behavior",
    "Min": "Packet-size behavior",
    "Max": "Packet-size behavior",
    "AVG": "Packet-size behavior",
    "Std": "Packet-size behavior",
    "Tot size": "Packet-size behavior",

    "IAT": "Timing/rate behavior",
    "Number": "Statistical flow behavior",
    "Magnitue": "Statistical flow behavior",
    "Radius": "Statistical flow behavior",
    "Covariance": "Statistical flow behavior",
    "Variance": "Statistical flow behavior",
    "Weight": "Statistical flow behavior"
}

# Add behavior group
shap_importance_shift["behavior_group"] = shap_importance_shift["feature"].map(feature_groups)

# Check missing groups
print("Missing group features:")
print(shap_importance_shift[shap_importance_shift["behavior_group"].isna()])

# Group contribution
shift_group_contribution = (
    shap_importance_shift
    .groupby("behavior_group")["mean_abs_shap"]
    .sum()
    .reset_index()
)

total_shap = shift_group_contribution["mean_abs_shap"].sum()

shift_group_contribution["percentage"] = (
    shift_group_contribution["mean_abs_shap"] / total_shap * 100
)

shift_group_contribution = shift_group_contribution.sort_values(
    by="percentage",
    ascending=False
)

print("Behavior-group SHAP contribution under shift:")
print(shift_group_contribution)


# Plot
plt.figure(figsize=(9, 5))
plt.barh(
    shift_group_contribution["behavior_group"],
    shift_group_contribution["percentage"]
)
plt.gca().invert_yaxis()
plt.xlabel("SHAP Contribution (%)")
plt.ylabel("Behavior Group")
plt.title("Behavior-Group SHAP Contribution Under Attack-Family Shift")
plt.tight_layout()
plt.show()